# 🧬 NeuralShield-v6.2 — EVREN MT Eğitim-Masum Üretimi (Seçenek P)
Bu defter **eğitim masum takviyesini** (25.600 EVREN-MT benign, 3'lü karışım) üretir ve Drive'a kaydeder.
GPU gerekmez (EVREN bir HTTP API). Süre ~20-30 dk.

**Önce:** Colab sol panel → 🔑 Secrets → `EVREN_API_KEY` ekle (evren_llm_...).

In [ ]:
# @title 1. Kurulum + Repo + EVREN anahtarı
import os
os.environ["TOKENIZERS_PARALLELISM"]="false"
!git clone -q https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git 2>/dev/null || (cd ai-soc-agent-lab && git pull -q)
%cd /content/ai-soc-agent-lab
!pip -q install datasets langdetect python-dotenv requests 2>/dev/null
try:
    from google.colab import userdata
    os.environ["EVREN_API_KEY"] = userdata.get("EVREN_API_KEY") or ""
except Exception:
    os.environ["EVREN_API_KEY"] = ""
if not os.environ.get("EVREN_API_KEY"):
    os.environ["EVREN_API_KEY"] = "BURAYA_EVREN_API_KEY_YAZ"
print("EVREN key ayarli mi:", bool(os.environ.get("EVREN_API_KEY")))
print("Calisma dizini:", os.getcwd())

In [ ]:
# @title 2. EVREN MT üretimi (v6.2) — ~20-30 dk
# 3'lu benign (%40 hard-neg + %30 roleplay + %30 genel) + mevcut saldirilar; sonucu data/v6_multilingual_supplement.json'a yazar
!python scripts/build_v6_multilingual_supplement.py --cap 20000

In [ ]:
# @title 3. Drive'a kaydet
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
D="/content/drive/MyDrive/NeuralShield_v6_data"
os.makedirs(D, exist_ok=True)
shutil.copy("data/v6_multilingual_supplement.json", D+"/")
if os.path.exists("data/_v6_mt_used.json"):
    shutil.copy("data/_v6_mt_used.json", D+"/")
r=__import__('json').load(open("data/v6_multilingual_supplement.json",encoding="utf-8"))
from collections import Counter
print("N=", len(r))
print("label:", dict(Counter(x['label'] for x in r)))
print("dil:", dict(Counter(x['language'] for x in r)))
print("Drive:", D)